In [7]:
import sys
import warnings
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, Ridge, ElasticNet

from src.config import TRAIN_PATH, TEST_PATH
from src.preprocessing import HousePricesPreprocessor
from src.validation import cross_validate

warnings.filterwarnings('ignore')

In [8]:
# Загрузка
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [9]:
y = np.log1p(train['SalePrice'])
test_ids = test['Id']
train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

# Preprocessing
preprocessor = HousePricesPreprocessor(scale=True)
X = preprocessor.fit_transform(train, np.expm1(y))
X_test = preprocessor.transform(test)

# Выбросы
outlier_idx = X[(X['GrLivArea'] > 4000) & (np.expm1(y) < 200000)].index
X = X.drop(outlier_idx)
y = y.drop(outlier_idx)

print(f"X: {X.shape}, y: {y.shape}")

X: (1460, 99), y: (1460,)


In [10]:
print("=== Ridge ===")
for alpha in [0.1, 1.0, 10.0, 50.0, 100.0]:
    print(f"\nalpha={alpha}:")
    cross_validate(Ridge(alpha=alpha), X, y)

=== Ridge ===

alpha=0.1:
RMSE по фолдам: [0.1246 0.1494 0.1254 0.116  0.1947]
Среднее: 0.1420 ± 0.0286

alpha=1.0:
RMSE по фолдам: [0.1242 0.1489 0.1253 0.116  0.1946]
Среднее: 0.1418 ± 0.0286

alpha=10.0:
RMSE по фолдам: [0.1229 0.1487 0.1245 0.1159 0.1935]
Среднее: 0.1411 ± 0.0284

alpha=50.0:
RMSE по фолдам: [0.1219 0.1496 0.1228 0.1165 0.1914]
Среднее: 0.1404 ± 0.0280

alpha=100.0:
RMSE по фолдам: [0.1218 0.1501 0.1219 0.117  0.1898]
Среднее: 0.1401 ± 0.0275


In [11]:
print("=== Ridge (больше alpha) ===")
for alpha in [100, 200, 500, 1000]:
    print(f"\nalpha={alpha}:")
    cross_validate(Ridge(alpha=alpha), X, y)

=== Ridge (больше alpha) ===

alpha=100:
RMSE по фолдам: [0.1218 0.1501 0.1219 0.117  0.1898]
Среднее: 0.1401 ± 0.0275

alpha=200:
RMSE по фолдам: [0.1225 0.1506 0.1218 0.118  0.1877]
Среднее: 0.1401 ± 0.0265

alpha=500:
RMSE по фолдам: [0.1263 0.1521 0.1245 0.1217 0.1849]
Среднее: 0.1419 ± 0.0241

alpha=1000:
RMSE по фолдам: [0.1329 0.155  0.1307 0.1275 0.1836]
Среднее: 0.1459 ± 0.0212


In [12]:
print("=== Lasso ===")
for alpha in [0.0001, 0.0005, 0.001, 0.005, 0.01]:
    print(f"\nalpha={alpha}:")
    cross_validate(Lasso(alpha=alpha), X, y)

=== Lasso ===

alpha=0.0001:
RMSE по фолдам: [0.1235 0.1487 0.1251 0.116  0.1947]
Среднее: 0.1416 ± 0.0287

alpha=0.0005:
RMSE по фолдам: [0.121  0.1474 0.124  0.1159 0.1935]
Среднее: 0.1404 ± 0.0287

alpha=0.001:
RMSE по фолдам: [0.1201 0.1472 0.1236 0.1165 0.192 ]
Среднее: 0.1399 ± 0.0282

alpha=0.005:
RMSE по фолдам: [0.1224 0.1458 0.1243 0.1204 0.1857]
Среднее: 0.1397 ± 0.0247

alpha=0.01:
RMSE по фолдам: [0.1277 0.1466 0.128  0.1255 0.1846]
Среднее: 0.1425 ± 0.0224


In [13]:
print("=== ElasticNet ===")
for alpha in [0.001, 0.005, 0.01]:
    for l1_ratio in [0.1, 0.5, 0.9]:
        print(f"\nalpha={alpha}, l1_ratio={l1_ratio}:")
        cross_validate(ElasticNet(alpha=alpha, l1_ratio=l1_ratio), X, y)

=== ElasticNet ===

alpha=0.001, l1_ratio=0.1:
RMSE по фолдам: [0.1233 0.1485 0.125  0.1159 0.1945]
Среднее: 0.1414 ± 0.0287

alpha=0.001, l1_ratio=0.5:
RMSE по фолдам: [0.121  0.1474 0.124  0.1159 0.1934]
Среднее: 0.1403 ± 0.0286

alpha=0.001, l1_ratio=0.9:
RMSE по фолдам: [0.1201 0.1471 0.1236 0.1163 0.1922]
Среднее: 0.1399 ± 0.0283

alpha=0.005, l1_ratio=0.1:
RMSE по фолдам: [0.1212 0.1476 0.1238 0.116  0.1928]
Среднее: 0.1403 ± 0.0284

alpha=0.005, l1_ratio=0.5:
RMSE по фолдам: [0.1211 0.1469 0.1234 0.1187 0.1891]
Среднее: 0.1398 ± 0.0266

alpha=0.005, l1_ratio=0.9:
RMSE по фолдам: [0.1221 0.1458 0.1241 0.1201 0.1861]
Среднее: 0.1396 ± 0.0250

alpha=0.01, l1_ratio=0.1:
RMSE по фолдам: [0.1208 0.1478 0.1235 0.1171 0.1913]
Среднее: 0.1401 ± 0.0278

alpha=0.01, l1_ratio=0.5:
RMSE по фолдам: [0.1225 0.1457 0.1243 0.1204 0.1853]
Среднее: 0.1396 ± 0.0246

alpha=0.01, l1_ratio=0.9:
RMSE по фолдам: [0.1266 0.1463 0.1271 0.1246 0.1841]
Среднее: 0.1418 ± 0.0226


Линейные модели: итоги
- LinearRegression:                    CV=0.1164  Kaggle=0.14316
- Ridge (alpha=100):                   CV=0.1401
- Lasso (alpha=0.005):                 CV=0.1397
- ElasticNet (alpha=0.005, l1=0.9):    CV=0.1396